<a href="https://colab.research.google.com/github/mncbl/Ap1VisaoComputacional/blob/main/AP1VisaoCOmputacional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install ultralytics opencv-python

In [10]:
from google.colab.patches import cv2_imshow

In [19]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict

# ==========================================
# 1. CONFIGURAÇÕES INICIAIS E MODELO
# ==========================================
model = YOLO('yolov8s.pt')
VEHICLE_CLASSES = [2, 3, 5, 7] # 2: Carro, 3: Moto, 5: Ônibus, 7: Caminhão
CONFIDENCE_THRESHOLD = 0.08 # Reduzido para capturar silhuetas fracas (motos)
IOU_THRESHOLD = 0.3 # Permite caixas sobrepostas (ex: moto passando do lado do carro)

# ==========================================
# 2. CONFIGURAÇÃO DE VÍDEO E SAÍDA
# ==========================================
video_path = '/content/drive/MyDrive/ap1visao/last_video.mp4'
cap = cv2.VideoCapture(video_path)

orig_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
orig_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
fps_efetivo = fps / 2

novo_width = 720
novo_height = int(orig_height * (novo_width / orig_width))

out = cv2.VideoWriter('resultado_expert_verde.mp4', cv2.VideoWriter_fourcc(*'mp4v'), int(fps_efetivo), (novo_width, novo_height))

# ==========================================
# 3. PIPELINE EXPERT: ESCALA DE CINZA
# ==========================================
def aplicar_filtro_grayscale_expert(frame):
    """
    Pipeline rigoroso focado em bordas e formas, ignorando crominância.
    """
    # 1. Correção Gamma: Suprime o estouro de luz (Blooming) dos faróis
    gamma = 2.5
    table = np.array([((i / 255.0) ** gamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
    frame_gamma = cv2.LUT(frame, table)

    # 2. Conversão para Escala de Cinza: Destrói o ruído de cor
    gray = cv2.cvtColor(frame_gamma, cv2.COLOR_BGR2GRAY)

    # 3. CLAHE Agressivo: Força o contraste local para revelar silhuetas escuras
    clahe = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8, 8))
    gray_clahe = clahe.apply(gray)

    # 4. Reconstrução BGR: O YOLO exige entrada com 3 canais (mesmo sendo cinza)
    frame_final = cv2.cvtColor(gray_clahe, cv2.COLOR_GRAY2BGR)

    # 5. Suavização (Denoise): Reduz o granulado do asfalto causado pelo CLAHE
    frame_final = cv2.GaussianBlur(frame_final, (3, 3), 0)

    return frame_final

# ==========================================
# 4. ENGENHARIA DE TRÁFEGO
# ==========================================
zona_faixa = (0, novo_width, int(novo_height * 0.70))
capacidade_faixa = 2400

historico_rotas = defaultdict(list)
veiculos_contados = set()

def calcular_los_vc(razao_vc):
    if razao_vc <= 0.2: return "LOS A (Livre)", (200, 255, 100)
    elif razao_vc <= 0.4: return "LOS B (Estavel)", (150, 255, 0)
    elif razao_vc <= 0.7: return "LOS C (Moderado)", (0, 255, 255)
    elif razao_vc <= 0.9: return "LOS D (Denso)", (0, 150, 255)
    else: return "LOS E/F (Critico)", (50, 50, 255)

# ==========================================
# 5. LOOP DE PROCESSAMENTO
# ==========================================
frame_count = 0
print("Iniciando processamento. IA analisando imagem limpa, desenhos aplicados no pós-processamento...")

while cap.isOpened():
    ret, frame_original = cap.read()
    if not ret: break

    frame_count += 1
    if frame_count % 2 != 0: continue

    frame_exibicao = cv2.resize(frame_original, (novo_width, novo_height))

    # Aplica o pipeline de cinza para a IA ver
    frame_filtrado = aplicar_filtro_grayscale_expert(frame_exibicao)
    overlay = frame_filtrado.copy()

    # ERRO CORRIGIDO: Inferência feita na imagem "LIMPA" (sem a linha desenhada atrapalhando)
    resultados = model.track(
        frame_filtrado,
        persist=True,
        tracker="bytetrack.yaml",
        conf=CONFIDENCE_THRESHOLD,
        iou=IOU_THRESHOLD,
        agnostic_nms=True,
        imgsz=736,
        verbose=False
    )[0]

    # AGORA SIM: Desenhamos a linha visual do sensor
    cv2.line(frame_filtrado, (zona_faixa[0], zona_faixa[2]), (zona_faixa[1], zona_faixa[2]), (0, 255, 0), 3)
    cv2.putText(frame_filtrado, "LINHA DE MEDICAO LOS", (10, zona_faixa[2] - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    if resultados.boxes is not None and resultados.boxes.id is not None:
        boxes = resultados.boxes.xyxy.cpu().numpy()
        clss = resultados.boxes.cls.cpu().numpy()
        ids = resultados.boxes.id.cpu().numpy()

        for box, cls, track_id in zip(boxes, clss, ids):
            if int(cls) in VEHICLE_CLASSES:
                x1, y1, x2, y2 = map(int, box)
                cx, cy = int((x1 + x2) / 2), int(y2)

                rota = historico_rotas[track_id]
                rota.append((cx, cy))
                if len(rota) > 30: rota.pop(0)

                # CORREÇÃO DE ESTILO: Tudo em Verde Neon
                cor_neon = (0, 255, 0)
                espessura_caixa = 3

                if len(rota) >= 2:
                    y_anterior = rota[-2][1]
                    y_atual = rota[-1][1]
                    x_atual = rota[-1][0]

                    dy = y_atual - y_anterior

                    # Contagem sentido único descendo
                    if dy > 1:
                        if y_anterior < zona_faixa[2] and y_atual >= zona_faixa[2] and zona_faixa[0] <= x_atual <= zona_faixa[1]:
                            veiculos_contados.add(track_id)

                # Renderização das caixas
                cv2.rectangle(frame_filtrado, (x1, y1), (x2, y2), cor_neon, espessura_caixa)

                # Se for Moto (cls 3), escreve MOTO em cima
                if int(cls) == 3:
                    cv2.putText(frame_filtrado, "MOTO", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, cor_neon, 2)

                # Renderização do rastro também em Verde Neon
                for i in range(1, len(rota)):
                    cv2.line(frame_filtrado, rota[i - 1], rota[i], cor_neon, 2)

    # --- MATEMÁTICA LOS ---
    tempo_decorrido_seg = max(1, (frame_count / 2) / fps_efetivo)
    vol_h = int((len(veiculos_contados) / tempo_decorrido_seg) * 3600)
    razao_vc = vol_h / capacidade_faixa
    los_txt, cor_los = calcular_los_vc(razao_vc)

    # --- DASHBOARD ---
    cv2.rectangle(overlay, (20, 20), (450, 140), (15, 15, 15), -1)
    cv2.addWeighted(overlay, 0.85, frame_filtrado, 0.15, 0, frame_filtrado)

    cv2.putText(frame_filtrado, 'VERDE', (30, 45), cv2.FONT_HERSHEY_DUPLEX, 0.6, (0, 255, 0), 1)
    cv2.line(frame_filtrado, (30, 55), (430, 55), (100, 100, 100), 1)
    cv2.putText(frame_filtrado, f'Volume de Fluxo: {vol_h} veiculos/hora', (30, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
    cv2.putText(frame_filtrado, f'Nivel de Servico: {los_txt}', (30, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.6, cor_los, 2)

    out.write(frame_filtrado)

cap.release()
out.release()
print("Processamento concluído com sucesso!")

Iniciando processamento. IA analisando imagem limpa, desenhos aplicados no pós-processamento...
Processamento concluído com sucesso!


For example, here we download and display a PNG image of the Colab logo: